# AMPS I · Week 14 — Lab
## Navier–Stokes: projection methods, a channel, a vortex and the DFG cylinder

| | |
|---|---|
| **Lecture sections** | Part A (Chorin, IPCS, CFL) · Part B (forms, solvers, drag and lift) · Part C (measurements) · Part D (LLM) |
| **Time** | ≈ 80 min |
| **You will** | check that IPCS reaches the exact Poiseuille state and Chorin does not · measure the temporal order of both on the Taylor–Green vortex · run the DFG 2D-3 benchmark on a coarse mesh and read drag, lift and the CFL number |

Taylor–Hood P2/P1 throughout; Crank–Nicolson in time with the convecting velocity extrapolated (Adams–Bashforth): one **linear** solve per sub-step.

**Reminder.** `%%fenicsx` cells run in a separate MPI process; ordinary cells have NumPy and matplotlib only. Data passes through `/tmp/fnx/`.

## ⚙️ Setup — run once per runtime
Installs DOLFINx 0.11 and registers the `%%fenicsx` cell magic. Re-run after every runtime restart.

In [ ]:
# ══════════════════════════════════════════════════════════════════
#  FEniCSx on Google Colab — run once per runtime (≈ 3–5 min)
#  https://github.com/SeoulTechPSE/fenicsx-colab
# ══════════════════════════════════════════════════════════════════
from google.colab import drive
import os, multiprocessing, subprocess
from pathlib import Path

if not os.path.ismount('/content/drive'):
    drive.mount('/content/drive')

REPO_URL = 'https://github.com/seoultechpse/fenicsx-colab.git'
REPO_DIR = Path('/content/fenicsx-colab')
if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)

# OpenMPI 5.x slot patch: Colab has 2 CPUs; allow up to 4 ranks (oversubscribed)
N_PROC = 4
with open('/root/hostfile', 'w') as f:
    f.write(f'localhost slots={N_PROC}\n')
os.environ['OMPI_MCA_rmaps_default_mapping_policy'] = 'slot:OVERSUBSCRIBE'
os.environ['PRTE_MCA_rmaps_default_mapping_policy'] = 'slot:OVERSUBSCRIBE'
os.environ['OMPI_MCA_osc'] = '^ucx'   # skip the UCX one-sided module (harmless start-up message)

DOLFINX_VERSION = '0.11'
USE_COMPLEX = False
USE_CLEAN   = False
opts = [f'--version {DOLFINX_VERSION}']
if USE_COMPLEX: opts.append('--complex')
if USE_CLEAN:   opts.append('--clean')
get_ipython().run_line_magic('run', f"{REPO_DIR / 'setup_fenicsx.py'} {' '.join(opts)}")

# Folder shared by %%fenicsx cells (computation) and ordinary cells (plotting)
os.makedirs('/tmp/fnx', exist_ok=True)

In [ ]:
%%fenicsx --info -np 2

## 1 · Channel flow: does the scheme reach the exact steady state?  *(lecture §A3–A4, §C2)*
Unit square, $\nu=1$, $\mathbf u=0$ on $y=0,1$, $p=8$ at $x=0$ and $p=0$ at $x=1$: steady solution $\mathbf u=(4y(1-y),0)$, $p=8(1-x)$ — in the Taylor–Hood space.
Read `channel` first: three sub-steps, which matrices are assembled once, and the boundary term $\int p\,\mathbf n\cdot\mathbf v\,ds$ (the pressure is prescribed on the in- and outlet).

In [ ]:
%%fenicsx -np 2
from mpi4py import MPI
from petsc4py import PETSc
import time
import numpy as np
import ufl
from dolfinx import fem, mesh
from dolfinx.fem.petsc import apply_lifting, assemble_matrix, assemble_vector, create_vector, set_bc

comm = MPI.COMM_WORLD

def say(*a):
    if comm.rank == 0:
        print(*a, flush=True)

def rate(E):
    E = np.asarray(E, dtype=float)
    return np.round(np.log2(E[:-1] / E[1:]), 2).tolist()

def lu(A):
    s = PETSc.KSP().create(comm); s.setOperators(A); s.setType("preonly")
    s.getPC().setType("lu"); s.getPC().setFactorSolverType("mumps")
    return s

def rhs(b, L, a=None, bcs=None):
    """Assemble L into b (zeroed first), lift and set the BCs of a, sum the ghost contributions."""
    with b.localForm() as loc:
        loc.set(0)
    assemble_vector(b, L)
    if a is not None:
        apply_lifting(b, [a], [bcs])
    b.ghostUpdate(addv=PETSc.InsertMode.ADD, mode=PETSc.ScatterMode.REVERSE)
    if bcs is not None:
        set_bc(b, bcs)
def channel(N, dt, T, scheme, nu_=1.0):
    """Unit square, u = 0 on y = 0, 1, p = 8 at x = 0, p = 0 at x = 1; exact steady u = (4y(1-y), 0), p = 8(1-x)."""
    msh = mesh.create_unit_square(comm, N, N)
    V = fem.functionspace(msh, ("Lagrange", 2, (2,))); Q = fem.functionspace(msh, ("Lagrange", 1))
    u, v = ufl.TrialFunction(V), ufl.TestFunction(V); p, q = ufl.TrialFunction(Q), ufl.TestFunction(Q)
    bcu = [fem.dirichletbc(np.zeros(2), fem.locate_dofs_geometrical(V, lambda x: np.isclose(x[1], 0) | np.isclose(x[1], 1)), V)]
    bcp = [fem.dirichletbc(8.0, fem.locate_dofs_geometrical(Q, lambda x: np.isclose(x[0], 0)), Q),
           fem.dirichletbc(0.0, fem.locate_dofs_geometrical(Q, lambda x: np.isclose(x[0], 1)), Q)]
    u_n, u_n1, u_s, u_ = (fem.Function(V) for _ in range(4)); p_n, p_ = fem.Function(Q), fem.Function(Q)
    k = fem.Constant(msh, dt); nu = fem.Constant(msh, nu_); n = ufl.FacetNormal(msh)
    U = 0.5 * (u + u_n)
    F1 = (ufl.dot((u - u_n) / k, v) + ufl.dot(ufl.dot(1.5 * u_n - 0.5 * u_n1, ufl.nabla_grad(U)), v) + nu * ufl.inner(ufl.grad(U), ufl.grad(v))) * ufl.dx
    if scheme == "ipcs":
        F1 += -p_n * ufl.div(v) * ufl.dx + p_n * ufl.dot(n, v) * ufl.ds         # grad p integrated by parts: the boundary term stays
        L2 = fem.form((ufl.dot(ufl.grad(p_n), ufl.grad(q)) - 1 / k * ufl.div(u_s) * q) * ufl.dx)
        L3 = fem.form((ufl.dot(u_s, v) - k * ufl.dot(ufl.grad(p_ - p_n), v)) * ufl.dx)
    else:
        L2 = fem.form(-1 / k * ufl.div(u_s) * q * ufl.dx)
        L3 = fem.form((ufl.dot(u_s, v) - k * ufl.dot(ufl.grad(p_), v)) * ufl.dx)
    a1, L1 = fem.form(ufl.lhs(F1)), fem.form(ufl.rhs(F1))
    a2 = fem.form(ufl.dot(ufl.grad(p), ufl.grad(q)) * ufl.dx); a3 = fem.form(ufl.dot(u, v) * ufl.dx)
    A1 = fem.petsc.create_matrix(a1); A2 = assemble_matrix(a2, bcs=bcp); A2.assemble(); A3 = assemble_matrix(a3); A3.assemble()
    b1, b2, b3 = create_vector(V), create_vector(Q), create_vector(V)
    s1, s2, s3 = lu(A1), lu(A2), lu(A3)
    x = ufl.SpatialCoordinate(msh); ue = ufl.as_vector([4 * x[1] * (1 - x[1]), 0]); pe = 8 * (1 - x[0])
    E = lambda e: np.sqrt(comm.allreduce(fem.assemble_scalar(fem.form(e * ufl.dx)), op=MPI.SUM))
    for _ in range(int(round(T / dt))):
        A1.zeroEntries(); assemble_matrix(A1, a1, bcs=bcu); A1.assemble(); s1.setOperators(A1)
        rhs(b1, L1, a1, bcu); s1.solve(b1, u_s.x.petsc_vec); u_s.x.scatter_forward()
        rhs(b2, L2, a2, bcp); s2.solve(b2, p_.x.petsc_vec); p_.x.scatter_forward()
        rhs(b3, L3); s3.solve(b3, u_.x.petsc_vec); u_.x.scatter_forward()
        u_n1.x.array[:] = u_n.x.array; u_n.x.array[:] = u_.x.array; p_n.x.array[:] = p_.x.array
    return E(ufl.inner(u_ - ue, u_ - ue)), E((p_ - pe) ** 2), E(ufl.div(u_) ** 2)

for dt in (0.01, 0.005, 0.0025):
    eu, ep, ed = channel(16, dt, 1.0, "ipcs")
    say(f"IPCS, dt = {dt}: at t = 1  |u - u_h| = {eu:.1e}, |p - p_h| = {ep:.1e}, |div u_h| = {ed:.1e}")

> **✏️ Your turn (5 min).** Run the same with `"chorin"`. Which error does not go to zero, how does it depend on $\Delta t$, and why (lecture §A3: what does Chorin's
> first step see of the pressure)?

In [ ]:
%%fenicsx -np 2
# TODO: Chorin on the channel

## 2 · Taylor–Green vortex: temporal order  *(lecture §A3–A4, §C1)*
$\mathbf u=(-\cos\pi x\sin\pi y,\ \sin\pi x\cos\pi y)\,e^{-2\pi^2\nu t}$, $\nu=0.01$, $T=1$, $N=32$, exact Dirichlet data on $\partial\Omega$. IPCS for four time steps:

In [ ]:
%%fenicsx -np 2
from mpi4py import MPI
from petsc4py import PETSc
import time
import numpy as np
import ufl
from dolfinx import fem, mesh
from dolfinx.fem.petsc import apply_lifting, assemble_matrix, assemble_vector, create_vector, set_bc

comm = MPI.COMM_WORLD

def say(*a):
    if comm.rank == 0:
        print(*a, flush=True)

def rate(E):
    E = np.asarray(E, dtype=float)
    return np.round(np.log2(E[:-1] / E[1:]), 2).tolist()

def lu(A):
    s = PETSc.KSP().create(comm); s.setOperators(A); s.setType("preonly")
    s.getPC().setType("lu"); s.getPC().setFactorSolverType("mumps")
    return s

def rhs(b, L, a=None, bcs=None):
    """Assemble L into b (zeroed first), lift and set the BCs of a, sum the ghost contributions."""
    with b.localForm() as loc:
        loc.set(0)
    assemble_vector(b, L)
    if a is not None:
        apply_lifting(b, [a], [bcs])
    b.ghostUpdate(addv=PETSc.InsertMode.ADD, mode=PETSc.ScatterMode.REVERSE)
    if bcs is not None:
        set_bc(b, bcs)
NU_TG = 0.01


def tg_exact(x, t, m=np):
    E = m.exp(-2 * m.pi ** 2 * NU_TG * t)
    u = (-m.cos(m.pi * x[0]) * m.sin(m.pi * x[1]) * E, m.sin(m.pi * x[0]) * m.cos(m.pi * x[1]) * E)
    p = -0.25 * (m.cos(2 * m.pi * x[0]) + m.cos(2 * m.pi * x[1])) * E ** 2
    return u, p


def taylor_green(N, dt, T, scheme):
    """Unit square, exact Dirichlet velocity on the boundary, one pressure value fixed at the origin. CN + AB2 convection.
    scheme 'ipcs': incremental pressure correction; 'chorin': non-incremental projection. Returns L2 errors of u and p (means removed)."""
    msh = mesh.create_unit_square(comm, N, N)
    V = fem.functionspace(msh, ("Lagrange", 2, (2,))); Q = fem.functionspace(msh, ("Lagrange", 1))
    u, v = ufl.TrialFunction(V), ufl.TestFunction(V); p, q = ufl.TrialFunction(Q), ufl.TestFunction(Q)
    u_n, u_n1, u_s, u_ = (fem.Function(V) for _ in range(4)); p_n, phi = fem.Function(Q), fem.Function(Q)
    u_n.interpolate(lambda x: np.vstack(tg_exact(x, 0.0)[0])); u_n1.interpolate(lambda x: np.vstack(tg_exact(x, -dt)[0]))
    if scheme == "ipcs":
        p_n.interpolate(lambda x: tg_exact(x, -0.5 * dt)[1])
    k = fem.Constant(msh, dt); nu = fem.Constant(msh, NU_TG)
    ub = fem.Function(V); pb = fem.Function(Q)
    msh.topology.create_connectivity(1, 2)
    bcu = [fem.dirichletbc(ub, fem.locate_dofs_topological(V, 1, mesh.exterior_facet_indices(msh.topology)))]
    bcp = [fem.dirichletbc(pb, fem.locate_dofs_geometrical(Q, lambda x: np.isclose(x[0], 0) & np.isclose(x[1], 0)))]
    U = 0.5 * (u + u_n)
    F1 = (ufl.dot((u - u_n) / k, v) + ufl.dot(ufl.dot(1.5 * u_n - 0.5 * u_n1, ufl.nabla_grad(U)), v)
          + nu * ufl.inner(ufl.grad(U), ufl.grad(v))) * ufl.dx
    if scheme == "ipcs":
        F1 -= p_n * ufl.div(v) * ufl.dx
    a1, L1 = fem.form(ufl.lhs(F1)), fem.form(ufl.rhs(F1))
    a2 = fem.form(ufl.inner(ufl.grad(p), ufl.grad(q)) * ufl.dx); L2 = fem.form(-1 / k * ufl.div(u_s) * q * ufl.dx)
    a3 = fem.form(ufl.dot(u, v) * ufl.dx); L3 = fem.form((ufl.dot(u_s, v) - k * ufl.dot(ufl.grad(phi), v)) * ufl.dx)
    A1 = fem.petsc.create_matrix(a1); A2 = assemble_matrix(a2, bcs=bcp); A2.assemble(); A3 = assemble_matrix(a3); A3.assemble()
    b1, b2, b3 = create_vector(V), create_vector(Q), create_vector(V)
    s1, s2, s3 = lu(A1), lu(A2), lu(A3)
    t = 0.0
    for _ in range(int(round(T / dt))):
        t += dt
        ub.interpolate(lambda x: np.vstack(tg_exact(x, t)[0]))
        tt = t
        pb.interpolate(lambda x: (tg_exact(x, tt - 0.5 * dt)[1] - tg_exact(x, tt - 1.5 * dt)[1]) if scheme == "ipcs" else tg_exact(x, tt)[1])
        A1.zeroEntries(); assemble_matrix(A1, a1, bcs=bcu); A1.assemble(); s1.setOperators(A1)
        rhs(b1, L1, a1, bcu); s1.solve(b1, u_s.x.petsc_vec); u_s.x.scatter_forward()
        rhs(b2, L2, a2, bcp); s2.solve(b2, phi.x.petsc_vec); phi.x.scatter_forward()
        rhs(b3, L3); s3.solve(b3, u_.x.petsc_vec); u_.x.scatter_forward()
        u_n1.x.array[:] = u_n.x.array; u_n.x.array[:] = u_.x.array
        p_n.x.array[:] = (p_n.x.array + phi.x.array) if scheme == "ipcs" else phi.x.array
    x = ufl.SpatialCoordinate(msh)
    ue, _ = tg_exact(x, t, ufl); ue = ufl.as_vector(ue)
    _, pe = tg_exact(x, t - 0.5 * dt if scheme == "ipcs" else t, ufl)     # IPCS (CN): pressure at t^{n+1/2}
    I = lambda e: comm.allreduce(fem.assemble_scalar(fem.form(e * ufl.dx)), op=MPI.SUM)
    pm, pem = I(p_n), I(pe)
    return np.sqrt(I(ufl.inner(u_n - ue, u_n - ue))), np.sqrt(I((p_n - pm - pe + pem) ** 2))

E = np.array([taylor_green(32, dt, 1.0, "ipcs") for dt in (0.1, 0.05, 0.025, 0.0125)])
say(f"IPCS: u {[f'{e:.2e}' for e in E[:, 0]]} rates {rate(E[:, 0])}")
say(f"      p {[f'{e:.2e}' for e in E[:, 1]]} rates {rate(E[:, 1])}")

> **✏️ Your turn (10 min).** The same for Chorin. Then explain (i) why the IPCS **pressure** rate collapses to $\approx0$ at the smallest $\Delta t$ (hint: what limits it
> when $\Delta t\to0$ at fixed $N$? Try $N=64$ for the last two time steps), (ii) the rates of Chorin.

In [ ]:
%%fenicsx -np 2
# TODO: Chorin rates; IPCS pressure at N = 64

## 3 · DFG 2D-3: flow past a cylinder  *(lecture §B3–B4, §C3)*
Channel $2.2\times0.41$, cylinder of radius 0.05 at $(0.2,0.2)$, $\nu=10^{-3}$, inflow $U(t)=1.5\sin(\pi t/8)$ at the channel centre ($\mathrm{Re}=100$ at $t=4$), $T=8$.
Coarse Gmsh mesh, $\Delta t=1/400$ (3200 steps; about 35 s on the instructor's machine, longer on Colab). Reference (Schäfer–Turek, John): $c_{D,\max}=2.9509$ at $t=3.936$,
$c_{L,\max}=0.4780$ at $t=5.693$, $\Delta p(8)=-0.1116$.

In [ ]:
%%fenicsx -np 2
from mpi4py import MPI
from petsc4py import PETSc
import time
import numpy as np
import ufl
from dolfinx import fem, mesh
from dolfinx.fem.petsc import apply_lifting, assemble_matrix, assemble_vector, create_vector, set_bc

comm = MPI.COMM_WORLD

def say(*a):
    if comm.rank == 0:
        print(*a, flush=True)

def rate(E):
    E = np.asarray(E, dtype=float)
    return np.round(np.log2(E[:-1] / E[1:]), 2).tolist()

def lu(A):
    s = PETSc.KSP().create(comm); s.setOperators(A); s.setType("preonly")
    s.getPC().setType("lu"); s.getPC().setFactorSolverType("mumps")
    return s

def rhs(b, L, a=None, bcs=None):
    """Assemble L into b (zeroed first), lift and set the BCs of a, sum the ghost contributions."""
    with b.localForm() as loc:
        loc.set(0)
    assemble_vector(b, L)
    if a is not None:
        apply_lifting(b, [a], [bcs])
    b.ghostUpdate(addv=PETSc.InsertMode.ADD, mode=PETSc.ScatterMode.REVERSE)
    if bcs is not None:
        set_bc(b, bcs)
import gmsh
from dolfinx import geometry
from dolfinx.io.gmsh import model_to_mesh

L_, H, C, R = 2.2, 0.41, (0.2, 0.2), 0.05
REF = dict(cd=2.950921575, t_cd=3.93625, cl=0.47795, t_cl=5.693125, dp=-0.1116)


def dfg_mesh(res_min_factor, lc_max):
    """Gmsh mesh of the DFG channel: cells of size r/res_min_factor at the cylinder, lc_max*H far away. Facet tags 2 inlet, 3 outlet, 4 walls, 5 cylinder."""
    gmsh.initialize(); gmsh.option.setNumber("General.Terminal", 0)
    if comm.rank == 0:
        rect = gmsh.model.occ.addRectangle(0, 0, 0, L_, H); obst = gmsh.model.occ.addDisk(C[0], C[1], 0, R, R)
        gmsh.model.occ.cut([(2, rect)], [(2, obst)]); gmsh.model.occ.synchronize()
        vol = gmsh.model.getEntities(dim=2); gmsh.model.addPhysicalGroup(2, [vol[0][1]], 1)
        tags = {2: [], 3: [], 4: [], 5: []}
        for bnd in gmsh.model.getBoundary(vol, oriented=False):
            com = gmsh.model.occ.getCenterOfMass(bnd[0], bnd[1])
            if np.allclose(com, [0, H / 2, 0]): tags[2].append(bnd[1])
            elif np.allclose(com, [L_, H / 2, 0]): tags[3].append(bnd[1])
            elif np.isclose(com[1], H) or np.isclose(com[1], 0): tags[4].append(bnd[1])
            else: tags[5].append(bnd[1])
        for tg, ent in tags.items():
            gmsh.model.addPhysicalGroup(1, ent, tg)
        d = gmsh.model.mesh.field.add("Distance"); gmsh.model.mesh.field.setNumbers(d, "EdgesList", tags[5])
        th = gmsh.model.mesh.field.add("Threshold"); gmsh.model.mesh.field.setNumber(th, "IField", d)
        gmsh.model.mesh.field.setNumber(th, "LcMin", R / res_min_factor); gmsh.model.mesh.field.setNumber(th, "LcMax", lc_max * H)
        gmsh.model.mesh.field.setNumber(th, "DistMin", R); gmsh.model.mesh.field.setNumber(th, "DistMax", 2 * H)
        gmsh.model.mesh.field.setAsBackgroundMesh(th)
        gmsh.option.setNumber("Mesh.Algorithm", 8)
        gmsh.model.mesh.generate(2); gmsh.model.mesh.setOrder(2); gmsh.model.mesh.optimize("Netgen")
    md = model_to_mesh(gmsh.model, comm, 0, gdim=2); gmsh.finalize()
    return md.mesh, md.facet_tags
MESHES = {"coarse": (3.0, 0.25), "medium": (6.0, 0.1)}

def dfg_run(level, dt, T=8.0):
    """IPCS (Crank-Nicolson + Adams-Bashforth convection) for DFG 2D-3: mu = 0.001, rho = 1, U(t) = 1.5 sin(pi t / 8) at the centre of the inlet.
    Returns a dict: t, CD, CL, dp (= p(0.15, 0.2) - p(0.25, 0.2) at T), cells, DOFs, wall time, max CFL, Krylov iterations per step."""
    msh, ft = dfg_mesh(*MESHES[level])
    V = fem.functionspace(msh, ("Lagrange", 2, (2,))); Q = fem.functionspace(msh, ("Lagrange", 1))
    out = dict(cells=msh.topology.index_map(2).size_global, dofs=V.dofmap.index_map.size_global * 2 + Q.dofmap.index_map.size_global)
    mu, rho = 0.001, 1.0
    tcur = [0.0]
    def inflow(x):
        val = np.zeros((2, x.shape[1])); val[0] = 4 * 1.5 * np.sin(tcur[0] * np.pi / 8) * x[1] * (H - x[1]) / H**2
        return val
    u_in = fem.Function(V); u_in.interpolate(inflow)
    bcu = [fem.dirichletbc(u_in, fem.locate_dofs_topological(V, 1, ft.find(2))),
           fem.dirichletbc(np.zeros(2), fem.locate_dofs_topological(V, 1, ft.find(4)), V),
           fem.dirichletbc(np.zeros(2), fem.locate_dofs_topological(V, 1, ft.find(5)), V)]
    bcp = [fem.dirichletbc(0.0, fem.locate_dofs_topological(Q, 1, ft.find(3)), Q)]
    u, v = ufl.TrialFunction(V), ufl.TestFunction(V); p, q = ufl.TrialFunction(Q), ufl.TestFunction(Q)
    u_, u_s, u_n, u_n1 = (fem.Function(V) for _ in range(4)); p_, phi = fem.Function(Q), fem.Function(Q)
    k = fem.Constant(msh, dt); muc = fem.Constant(msh, mu); rhoc = fem.Constant(msh, rho)
    U = 0.5 * (u + u_n)                                   # Crank-Nicolson
    F1 = (rhoc / k * ufl.dot(u - u_n, v) + rhoc * ufl.inner(ufl.dot(1.5 * u_n - 0.5 * u_n1, ufl.nabla_grad(U)), v)
          + muc * ufl.inner(ufl.grad(U), ufl.grad(v)) - p_ * ufl.div(v)) * ufl.dx
    a1, L1 = fem.form(ufl.lhs(F1)), fem.form(ufl.rhs(F1))
    a2 = fem.form(ufl.dot(ufl.grad(p), ufl.grad(q)) * ufl.dx); L2 = fem.form(-rhoc / k * ufl.div(u_s) * q * ufl.dx)
    a3 = fem.form(rhoc * ufl.dot(u, v) * ufl.dx); L3 = fem.form((rhoc * ufl.dot(u_s, v) - k * ufl.dot(ufl.grad(phi), v)) * ufl.dx)
    A1 = fem.petsc.create_matrix(a1)
    A2 = assemble_matrix(a2, bcs=bcp); A2.assemble(); A3 = assemble_matrix(a3); A3.assemble()
    b1, b2, b3 = create_vector(V), create_vector(Q), create_vector(V)
    s1 = PETSc.KSP().create(comm); s1.setOperators(A1); s1.setType("bcgs"); s1.getPC().setType("jacobi")
    s2 = PETSc.KSP().create(comm); s2.setOperators(A2); s2.setType("minres"); s2.getPC().setType("hypre"); s2.getPC().setHYPREType("boomeramg")
    s3 = PETSc.KSP().create(comm); s3.setOperators(A3); s3.setType("cg"); s3.getPC().setType("sor")
    for s in (s1, s2, s3):
        s.setTolerances(rtol=1e-8)
    n = -ufl.FacetNormal(msh)                               # normal pointing out of the cylinder
    dS = ufl.Measure("ds", domain=msh, subdomain_data=ft, subdomain_id=5)
    u_t = ufl.inner(ufl.as_vector((n[1], -n[0])), u_)
    drag = fem.form(2 / 0.1 * (muc / rhoc * ufl.inner(ufl.grad(u_t), n) * n[1] - p_ * n[0]) * dS)
    lift = fem.form(-2 / 0.1 * (muc / rhoc * ufl.inner(ufl.grad(u_t), n) * n[0] + p_ * n[1]) * dS)
    W0 = fem.functionspace(msh, ("DG", 0)); speed = fem.Function(W0)
    speed_expr = fem.Expression(ufl.sqrt(ufl.dot(u_, u_)), W0.element.interpolation_points)
    nloc = msh.topology.index_map(2).size_local
    hK = msh.h(2, np.arange(nloc, dtype=np.int32))   # ufl.CellDiameter does not compile on this curved mesh
    TT, CD, CL = [], [], []; its = np.zeros(3); cflmax = 0.0
    t0 = time.perf_counter(); nsteps = int(round(T / dt))
    for i in range(nsteps):
        tcur[0] += dt; u_in.interpolate(inflow)
        A1.zeroEntries(); assemble_matrix(A1, a1, bcs=bcu); A1.assemble()
        rhs(b1, L1, a1, bcu); s1.solve(b1, u_s.x.petsc_vec); u_s.x.scatter_forward()
        rhs(b2, L2, a2, bcp); s2.solve(b2, phi.x.petsc_vec); phi.x.scatter_forward()
        p_.x.petsc_vec.axpy(1.0, phi.x.petsc_vec); p_.x.scatter_forward()
        rhs(b3, L3); s3.solve(b3, u_.x.petsc_vec); u_.x.scatter_forward()
        its += [s1.getIterationNumber(), s2.getIterationNumber(), s3.getIterationNumber()]
        u_n1.x.array[:] = u_n.x.array; u_n.x.array[:] = u_.x.array
        TT.append(tcur[0])
        CD.append(comm.allreduce(fem.assemble_scalar(drag), op=MPI.SUM)); CL.append(comm.allreduce(fem.assemble_scalar(lift), op=MPI.SUM))
        if i % 20 == 0:
            speed.interpolate(speed_expr); cflmax = max(cflmax, comm.allreduce((dt * speed.x.array[:nloc] / hK).max(), op=MPI.MAX))
    out["walltime"] = comm.allreduce(time.perf_counter() - t0, op=MPI.MAX)
    tree = geometry.bb_tree(msh, 2)
    pts = np.array([[0.15, 0.2, 0.0], [0.25, 0.2, 0.0]])
    cells = geometry.compute_colliding_cells(msh, geometry.compute_collisions_points(tree, pts), pts)
    loc = [p_.eval(pts[j], cells.links(j)[:1])[0] if len(cells.links(j)) > 0 else -np.inf for j in range(2)]
    pa = [comm.allreduce(loc[j], op=MPI.MAX) for j in range(2)]          # only the rank that owns the point has a value
    out.update(t=np.array(TT), CD=np.array(CD), CL=np.array(CL), dp=pa[0] - pa[1], its=its / nsteps, cfl=cflmax)
    return out

def summary(tag, r):
    i, j = r["CD"].argmax(), r["CL"].argmax()
    say(f"{tag}: {r['cells']} cells, {r['dofs']} DOFs, {r['walltime']:.0f} s, max CFL {r['cfl']:.2f}, its/step {np.round(r['its'], 1).tolist()} | "
        f"CD_max {r['CD'][i]:.4f} at t = {r['t'][i]:.3f} ({100 * (r['CD'][i] / REF['cd'] - 1):+.1f} %) | "
        f"CL_max {r['CL'][j]:.4f} at t = {r['t'][j]:.3f} ({100 * (r['CL'][j] / REF['cl'] - 1):+.1f} %) | dp(8) {r['dp']:.4f} (ref {REF['dp']})")

r = dfg_run("coarse", 1 / 400)
summary("coarse, dt = 1/400", r)
if comm.rank == 0:
    np.savez("/tmp/fnx/w14_dfg_coarse.npz", t=r["t"], CD=r["CD"], CL=r["CL"])

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

d = np.load("/tmp/fnx/w14_dfg_coarse.npz")
fig, axs = plt.subplots(1, 2, figsize=(11, 3.6))
axs[0].plot(d["t"], d["CD"]); axs[0].plot(3.93625, 2.950921575, "k*", ms=12); axs[0].set_title("drag coefficient")
axs[1].plot(d["t"], d["CL"]); axs[1].plot(5.693125, 0.47795, "k*", ms=12); axs[1].set_title("lift coefficient")
for ax in axs:
    ax.set_xlabel("t")
fig.tight_layout(); plt.show()

> **✏️ Your turn (10 min).** (a) From the printed maximal CFL number and the mesh: what is $\Delta t$ in units of the "CFL = 1" time step? (b) The cell Reynolds number
> $\mathrm{Re}_h=|\mathbf u|h/\nu$ near the cylinder ($|\mathbf u|\approx1.5$, $h\approx r/3$): compare with the Péclet threshold of Week 13. (c) Why does the coarse mesh get $c_{D,\max}$
> within a few per cent but the lift peak late and too small?

> *your answers to (a)–(c)*

## 4 · LLM-assisted coding: live demo  *(lecture §D1)*
Prompt (paste the API reference sheet first):

```text
Context: [API reference sheet]
Task: DOLFINx 0.11, one %%fenicsx -np 2 cell. Taylor-Green vortex on the unit square, nu = 0.01,
u = (-cos(pi x) sin(pi y), sin(pi x) cos(pi y)) exp(-2 pi^2 nu t), p = -(cos(2 pi x) + cos(2 pi y))/4 exp(-4 pi^2 nu t),
exact velocity on the boundary. Taylor-Hood P2/P1, incremental pressure correction (IPCS) with Crank-Nicolson and Adams-Bashforth
convection. For dt = 0.1, 0.05, 0.025 print the L2 velocity error at T = 1 and the observed order, on rank 0.
```
Paste the result below **unchanged**, run it, then go through the checklist.

In [ ]:
%%fenicsx -np 2
# TODO: paste the %%fenicsx part your LLM produced (unchanged) and run it

**Verification checklist** — checks with a known answer

- [ ] Convection written as $(\mathbf w\cdot\nabla)\mathbf u$: `dot(w, nabla_grad(u))` (or `dot(grad(u), w)`) — not `dot(nabla_grad(u), w)`
- [ ] A steady exact solution (Poiseuille) is reproduced to round-off by the time stepping
- [ ] Velocity order 2 with IPCS + Crank–Nicolson on a mesh fine enough that the spatial error is below the temporal one
- [ ] The pressure constant is fixed when the velocity is prescribed on the whole boundary
- [ ] Same numbers with `-np 1` and `-np 2`